In [ ]:
import json
import numpy as np
import pandas as pd
import seaborn as sns
import simpy
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.stats import lognorm, kstest, norm
from tqdm.notebook import tqdm
from uuid import uuid4

In [ ]:
class BaseSim:
    def __init__(self, env, rng=np.random.default_rng(), output_log=False, **kwargs):
        self.env = env
        self.rng = rng
        self.output_log = output_log

    def now(self):
        return round(self.env.now, 2)
    
    def print_log(self, msg):
        if self.output_log:
            print(f"Time {self.now()}: {msg}")


class Broker(BaseSim):
    def __init__(self, env, latency_params=None, audit_prob=0.0, **kwargs):
        super().__init__(env, **kwargs)
        self.proxies = {}
        self.messages = {}
        self.next_proxy = -1
        self.latency_params = latency_params
        self.audit_prob = audit_prob

    def register_proxy(self, proxy):
        self.proxies[proxy.id] = proxy

    def get_best_proxy(self):
        yield self.sim_db_time()
        starting_proxy = self.next_proxy
        while True:
            self.next_proxy = (self.next_proxy + 1) % len(self.proxies)
            if self.next_proxy == starting_proxy:
                self.next_proxy = (self.next_proxy + 1) % len(self.proxies)
                break
            audit_messages = [msg for msg in self.messages.values()
                if msg["proxy_id"] == self.next_proxy and msg["audit"] and not msg["finished"]
            ]
            if len(audit_messages) == 0:
                break
            
        return self.next_proxy

    def sim_trip_time(self):
        latency = 1
        if self.latency_params is not None:
            latency = lognorm.rvs(*self.latency_params, random_state=self.rng)
        return self.env.timeout(latency)

    def sim_db_time(self):
        return self.env.timeout(self.rng.uniform(low=0.1, high=0.3))

    def user_send_prompt(self, user_id, proxy_id, prompt_tokens, output_tokens):
        assert proxy_id in self.proxies
        self.messages[user_id] = {
            "proxy_id": proxy_id,
            "time": self.now(),
            "prompt_tokens": prompt_tokens,
            "output_tokens": output_tokens,
            "finished": False,
            "audit": False,
        }
        yield self.sim_trip_time()
        yield self.sim_db_time()

    def user_retrieve_response(self, user_id):
        is_finished = False
        if user_id in self.messages:
            is_finished = self.messages[user_id]["finished"]
            if is_finished:
                del self.messages[user_id]
        yield self.sim_trip_time()
        yield self.sim_db_time()
        return is_finished

    def proxy_retrieve_prompts(self, proxy_id):
        yield self.sim_trip_time()
        yield self.sim_db_time()
        audit_messages = [
            msg
            for msg in self.messages.values()
            if msg["proxy_id"] == proxy_id and msg["audit"] and msg["audit_status"] != "SUCCESS"
        ]
        if len(audit_messages) > 0:
            return []

        regular_messages = [
            (user_id, msg["prompt_tokens"], msg["output_tokens"])
            for user_id, msg in self.messages.items()
            if msg["proxy_id"] == proxy_id and not msg["finished"] and not msg["audit"]
        ]        
        return regular_messages

    def proxy_send_response(self, user_id):
        yield self.sim_trip_time()
        yield self.sim_db_time()
        msg = self.messages[user_id]
        msg["finished"] = True

        if self.rng.uniform() < self.audit_prob:
            self.print_log(f"Broker requesting audit proxy {msg['proxy_id']}")
            msg["audit"] = True
            msg["audit_status"] = "STARTED"
            return "AUDIT"

    def proxy_send_audit_proof(self, user_id):
        msg = self.messages[user_id]
        if msg["audit"] and msg["audit_status"] == "STARTED":
            msg["audit_status"] = "SUCCEEDED"
            msg["finished"] = True
        
        yield self.sim_trip_time()
        yield self.sim_db_time()


class Chatbot(BaseSim):
    def __init__(self, env, latency_params, prefill_per_sec=500, gen_per_sec=125, **kwargs):
        super().__init__(env, **kwargs)
        self.prefill_per_sec = prefill_per_sec
        self.gen_per_sec = gen_per_sec
        self.latency_params = latency_params

    def query(self, prompt_tokens, output_tokens):
        latency = 0
        if self.latency_params is not None:
            latency = lognorm.rvs(*self.latency_params, random_state=self.rng)
        return self.env.timeout(
            latency
            + int(prompt_tokens / self.prefill_per_sec)
            + int(output_tokens / self.gen_per_sec)
        )


class Proxy(BaseSim):
    def __init__(self, env, id, broker, chatbot, rate_limit=10, poll_interval=1, **kwargs):
        super().__init__(env, **kwargs)
        self.id = id
        self.broker = broker
        self.chatbot = chatbot
        self.rate_limit = rate_limit
        self.rate_limiter = simpy.Container(env, init=rate_limit, capacity=rate_limit)
        self.poll_interval = poll_interval
        broker.register_proxy(self)
        self.run_proc = env.process(self.run())
        self.reset_limit_proc = env.process(self.reset_rate_limit())

    def sim_gui_time(self):
        return self.env.timeout(1)

    def run(self):
        while True:
            # self.print_log(f"Proxy {self.id} checking for messages")
            prompts = yield self.env.process(
                self.broker.proxy_retrieve_prompts(self.id)
            )
            if len(prompts) > 0:
                self.print_log(
                    f"Proxy {self.id} got {len(prompts)} messages"
                )
                for user_id, prompt_tokens, output_tokens in prompts:
                    yield self.rate_limiter.get(1)
                    self.print_log(
                        f"Proxy {self.id} working on message {user_id}"
                    )
                    yield self.sim_gui_time()
                    yield self.chatbot.query(prompt_tokens, output_tokens)
                    self.print_log(
                        f"Proxy {self.id} sending response to message {user_id}"
                    )
                    yield self.sim_gui_time()
                    msg = yield self.env.process(self.broker.proxy_send_response(user_id))
                    self.print_log(
                        f"Proxy {self.id} sent response to message {user_id}"
                    )

                    if msg == "AUDIT":
                        self.print_log(f"Proxy {self.id} starting audit")
                        yield self.env.timeout(int(self.rng.uniform(80, 100)))
                        yield self.env.process(self.broker.proxy_send_audit_proof(user_id))
                        self.print_log(f"Proxy {self.id} finished audit")
            yield self.env.timeout(self.poll_interval)

    def reset_rate_limit(self):
        while True:
            if self.now() % 3600 == 0 and self.rate_limiter.level < self.rate_limit:
                self.print_log(f"Proxy {self.id} resetting rate limit")
                yield self.rate_limiter.put(self.rate_limit - self.rate_limiter.level)
            else:
                yield self.env.timeout(3600)


class User(BaseSim):
    def __init__(self, env, id, broker, poll_interval=1, prompt_tokens=100, output_tokens=1000, mean_think_time=300, **kwargs):
        super().__init__(env, **kwargs)
        self.id = id
        self.broker = broker
        self.poll_interval = poll_interval
        self.prompt_tokens = prompt_tokens
        self.output_tokens = output_tokens
        self.mean_think_time = mean_think_time
        self.log = []
        env.process(self.chat())

    def chat(self):
        while True:
            uid = uuid4()
            best_proxy_id = yield self.env.process(self.broker.get_best_proxy())
            start = self.now()
            self.print_log(
                f"User {self.id} sending message {uid} to proxy {best_proxy_id}"
            )
            yield self.env.process(
                self.broker.user_send_prompt(uid, best_proxy_id, self.prompt_tokens, self.output_tokens)
            )
            self.print_log(
                f"User {self.id} sent message {uid} to proxy {best_proxy_id}"
            )

            while True:
                res = yield self.env.process(self.broker.user_retrieve_response(uid))
                if res:
                    end = self.now()
                    self.print_log(
                        f"User {self.id} got response to message {uid} ({round(end-start, 2)} sec)"
                    )
                    self.log.append({"start": start, "end": end})
                    break
                else:
                    yield self.env.timeout(self.poll_interval)

            # Think for a bit
            think_time = np.random.exponential(self.mean_think_time)
            yield self.env.timeout(think_time)  


class UserGenerator(BaseSim):
    def __init__(self, env, broker, num_users=1, mean_inter_arrival_time=60, **kwargs):
        super().__init__(env, **kwargs)
        self.broker = broker
        self.num_users = num_users
        self.users = []
        self.mean_inter_arrival_time = mean_inter_arrival_time
        self.kwargs = kwargs
        env.process(self.generate())

    def generate(self):
        for i in range(self.num_users):
            inter_arrival_time = int(
                np.random.exponential(self.mean_inter_arrival_time)
            )
            yield self.env.timeout(inter_arrival_time)
            self.print_log(f"User {i} arrived")
            self.users.append(User(self.env, i, self.broker, **self.kwargs))

    def get_user_stats(self, plot=False):
        df = pd.DataFrame.from_dict([log for user in self.users for log in user.log])
        time_diff = df["end"] - df["start"]

        if plot:
            plt.hist(time_diff, bins='auto')

        return {
            "mean_wait_time": round(time_diff.mean(), 2),
            **{
                f"p{round(q * 100, 1)}_wait_time": round(time_diff.quantile(q), 2)
                for q in [0.025, 0.5, 0.9, 0.95, 0.975, 0.99]
            },
        }


def fit_latency(filepath, plot=False):
    with open(filepath, "r") as f:
        raw_latency = f.readlines()
    latency = []
    for conn in raw_latency:
        for ping in conn.split(";"):
            a, b, c = ping.split(",")
            a = int(a.split(":")[1])
            c = int(c.split(":")[1])
            latency.append((c - a) / 1_000_000_000)    

    shape, loc, scale = lognorm.fit(latency, floc=0)

    if plot:
        D, p_value = kstest(latency, 'lognorm', args=(shape, loc, scale))
        print(f"K-S test: D={D:.4f}, p={p_value:.4f}")

        x = np.linspace(min(latency), max(latency), 200)
        pdf = lognorm.pdf(x, shape, loc, scale)

        plt.figure(figsize=(8,5))
        plt.hist(latency, bins=20, density=True, alpha=0.6, color='lightblue', label='Measured data')
        plt.plot(x, pdf, 'r-', lw=2, label='Lognormal fit')
        plt.xlabel('Latency (ms)')
        plt.ylabel('Density')
        plt.title('Lognormal Fit to Network Latency')
        plt.legend()
        plt.show()

    return shape, loc, scale


def run_sim(
    num_proxies=1,
    num_users=1,
    ac_latency_params=None,
    normal_latency_params=None,
    rate_limit=20,
    duration=3600,
    audit_prob=0.0,
    output_log=False,
):
    assert ac_latency_params is not None and normal_latency_params is not None
    env = simpy.Environment()
    broker = Broker(env, ac_latency_params, audit_prob=audit_prob, output_log=output_log)
    chatbot = Chatbot(env, normal_latency_params, output_log=output_log)
    proxies = [
        Proxy(env, i, broker, chatbot, rate_limit=rate_limit, output_log=output_log)
        for i in range(num_proxies)
    ]
    user_generator = UserGenerator(
        env,
        broker,
        num_users=num_users,
        mean_inter_arrival_time=5,
        prompt_tokens=100,
        output_tokens=1000,
        mean_think_time=300,
        output_log=output_log,
    )
    env.run(until=duration)
    stats = user_generator.get_user_stats()
    return stats

In [ ]:
def fit_lognormal_leastsq(q25, q50, q75):
    # Define available quantiles and their approximate probabilities
    probs = [0.25, 0.50, 0.75]
    qvals = [q25, q50, q75]

    # Convert to log-space
    y = np.log(qvals)
    z = norm.ppf(probs)

    # Define residuals function
    def residuals(params):
        mu, sigma = params
        return y - (mu + sigma * z)

    # Initial guess from quartiles
    mu0 = np.log(q50)
    sigma0 = (np.log(q75) - np.log(q25)) / (norm.ppf(0.75) - norm.ppf(0.25))

    # Solve least squares
    res = least_squares(residuals, x0=[mu0, sigma0])

    mu, sigma = res.x

    # print("Least-squares lognormal fit:")
    # print(f"  mu    = {mu:.6f}")
    # print(f"  sigma = {sigma:.6f}")
    # print(f"  mean (expected value) = {np.exp(mu + sigma**2 / 2):.6f}")
    # print(f"  variance              = {(np.exp(sigma**2) - 1) * np.exp(2*mu + sigma**2):.6f}")
    # print(f"  residual RMS (log-space) = {np.sqrt(np.mean(res.fun**2)):.6e}")

    return mu, sigma, np.sqrt(np.mean(res.fun**2))

def plot_lognormal_fit(q25, q50, q75, min_val=None, max_val=None):
    """
    Fit and plot the estimated lognormal distribution against empirical points.
    """
    mu, sigma = fit_lognormal_leastsq(q25, q50, q75)

    dist = lognorm(s=sigma, scale=np.exp(mu))

    # Compute estimated quartiles
    est_q25, est_q50, est_q75 = dist.ppf([0.25, 0.5, 0.75])

    # X-axis range
    x_min = min_val if min_val else q25 / 5
    x_max = max_val if max_val else q75 * 5
    x = np.linspace(x_min, x_max, 500)
    pdf = dist.pdf(x)

    # --- Plot PDF ---
    plt.figure(figsize=(9, 5))
    plt.plot(x, pdf, label='Fitted Lognormal PDF', lw=2, color='black')

    # Empirical quartiles and non-outlier bounds
    for label, val, color in [
        ('min (non-outlier)', min_val, 'gray'),
        ('q25 (empirical)', q25, 'blue'),
        ('median (empirical)', q50, 'green'),
        ('q75 (empirical)', q75, 'red'),
        ('max (non-outlier)', max_val, 'gray')
    ]:
        if val is not None:
            plt.axvline(val, color=color, linestyle='--', alpha=0.7, label=label)

    # Estimated quartiles (from fitted mu, sigma)
    for label, val, color in [
        ('q25 (estimated)', est_q25, 'blue'),
        ('median (estimated)', est_q50, 'green'),
        ('q75 (estimated)', est_q75, 'red')
    ]:
        plt.axvline(val, color=color, linestyle='-', lw=1.8, alpha=0.9)
        plt.text(val, plt.ylim()[1]*0.9, label, rotation=90,
                 va='top', ha='right', fontsize=8, color=color)

    plt.title(f"Fitted Lognormal Distribution\nμ={mu:.3f}, σ={sigma:.3f}")
    plt.xlabel("Value")
    plt.ylabel("PDF")
    plt.legend(loc="upper right", fontsize=8)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("\nEmpirical vs Estimated Quartiles:")
    print(f"  q25: empirical={q25:.3f}, estimated={est_q25:.3f}")
    print(f"  q50: empirical={q50:.3f}, estimated={est_q50:.3f}")
    print(f"  q75: empirical={q75:.3f}, estimated={est_q75:.3f}")

def fit_latency_from_quantiles(filepath):
    df = pd.read_csv(filepath)
    mus = []
    sigmas = []
    residuals = []
    for _, row in df.iterrows():
        mu, sigma, residual = fit_lognormal_leastsq(row["q1"] / 1000.0, row["md"] / 1000.0, row["q3"] / 1000.0)
        mus.append(mu)
        sigmas.append(sigma)
        residuals.append(residual)
    df["mu"] = mus
    df["sigma"] = sigmas
    df["residual"] = residuals
    return df

In [ ]:
tor_params_est = fit_latency_from_quantiles("onionperf-latencies-2025-08-06-2025-11-04-onion.csv")
# tor_server = "op-us8a"
tor_server = "op-hk8a"
mu, sigma = tor_params_est[tor_params_est["source"] == tor_server].iloc[-1][["mu", "sigma"]].to_numpy()
tor_latency_params = (sigma, 0, np.exp(mu))
# tor_latency_params = fit_latency("latency_tor.txt", plot=False)
mu, sigma = tor_params_est[tor_params_est["source"] == "op-de8a"].iloc[-1][["mu", "sigma"]].to_numpy()
vpn_latency_params = fit_latency("latency_vpn.txt", plot=False)
pr_latency_params = (vpn_latency_params[0], 0, np.exp(mu))
normal_latency_params = fit_latency("latency_normal.txt", plot=False)
run_sim(
    num_proxies=1,
    num_users=1,
    ac_latency_params=pr_latency_params,
    normal_latency_params=normal_latency_params,
    duration=3600 * 24,
    audit_prob=0.0,
    output_log=False,
)

In [ ]:
for audit_prob in [0.0, 0.25, 0.5, 0.75, 1.0]:
    df = []
    num_proxies = 30

    for num_users in [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]:
        print(f"Simulating for {num_users} users")
        res = run_sim(
            num_proxies=num_proxies,
            num_users=num_users,
            ac_latency_params=vpn_latency_params,
            normal_latency_params=normal_latency_params,
            duration=3600 * 24,
            audit_prob=audit_prob,
            output_log=False,
        )
        print(res)
        df.append({
            "num_proxies": num_proxies,
            "num_users": num_users,
            "audit_prob": audit_prob,
            **res,
        })
    df = pd.DataFrame.from_dict(df)
    df.to_csv(f"vpn_sim_audit_{audit_prob}.csv", index=False)

In [ ]:
maximum_rpm = 20
for p in [0.05, 0.95, 1.0, 0.0, 0.25, 0.5, 0.75]:
    df = pd.read_csv(f"vpn_sim_audit_{p}.csv")
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(6, 4))
    palette = sns.color_palette(n_colors=4)
    ratios = df["num_users"] / df["num_proxies"]
    rpm = df["num_users"] * 12 // 60
    effective_rpm = round(maximum_rpm * (1 - p))
    if effective_rpm == 0:
        load_factor = rpm
    else:
        load_factor = np.round(rpm / effective_rpm, 2)
    # rpm = (df["num_users"] * 12 * 10 * (1 + p) // 60) / (20 * 100 / 60)
    # rpm = np.round(rpm, 2)

    for i, q in enumerate([50, 90, 95, 99]):
        line = df[f"p{q}.0_wait_time"]
        color = palette[i]
        plt.plot(load_factor, line, label=f"p{q}", color=color, marker='.')

    # Customize the plot
    # plt.xlabel("User : Proxy ratio")
    # plt.xlabel("Requests per minute")
    # plt.ylabel("Latency overhead (seconds)")
    plt.xticks(load_factor, labels=load_factor)
    plt.tick_params(labelsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, linestyle="--", color="lightgray")
    plt.tight_layout(pad=0)
    # plt.savefig(f"vpn_sim_audit_{p}.pdf", format="pdf")
    plt.show()

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(6, 4))
palette = sns.color_palette(n_colors=5)

plt.vlines(10, -50, 2500, color="gray")
# plt.text(24.5, 2200, "Max capacity")
for i, p in enumerate([0.0, 0.25, 0.5, 0.75, 1.0]):
    df = pd.read_csv(f"vpn_sim_audit_{p}.csv")
    ratios = df["num_users"] / df["num_proxies"]
    rpm = df["num_users"] * 12 // 60

    
    line = (df[f"p99.0_wait_time"])
    color = palette[i]
    plt.plot(rpm, line, label=f"{p}", color=color, marker='.')

    # Customize the plot
    # plt.xlabel("User : Proxy ratio")
# plt.xlabel("Requests per minute")
# plt.ylabel("p99 latency overhead (seconds)")
plt.ylim(-50, 2400)
plt.xticks(rpm, labels=rpm)
plt.tick_params(labelsize=12)
plt.legend(title="Audit chance", fontsize=12)
plt.grid(True, linestyle="--", color="lightgray")
plt.tight_layout(pad=0)
plt.savefig(f"vpn_sim_p99_audits.pdf", format="pdf")
plt.show()

In [ ]:
with open("./private_relay/browsertime-results-with-pr/browsertime.json") as f:
    with_pr = json.load(f)
with open("./private_relay/browsertime-results-without-pr/browsertime.json") as f:
    without_pr = json.load(f)

with_pr = [o["timings"]["ttfb"] for o in with_pr[0]["browserScripts"]][1:]
without_pr = [o["timings"]["ttfb"] for o in without_pr[0]["browserScripts"]][1:]

plt.hist(with_pr, bins="auto")
plt.show()

print(np.mean(with_pr), np.std(with_pr), np.median(with_pr))
print(np.mean(without_pr), np.std(without_pr), np.median(without_pr))